<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/Flux%20Klein%204B%20Outpainting%20Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background-color: #161b22; padding: 20px; border-radius: 12px; border-left: 6px solid #2ecc71; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
  <h1 style="color: #ffffff; margin: 0; font-family: sans-serif; font-weight: 700; letter-spacing: 1px;">
    ⚡ Flux Klein 4B Outpaint <span style="font-size: 0.6em; color: #2ecc71; vertical-align: middle; background: #2ecc7122; padding: 2px 8px; border-radius: 6px;">Turbo Engine</span>
  </h1>
</div>

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**
*Subscribe for more AI tutorials, workflows, and optimization tips!*

**Quick Start Guide:**
This notebook lets you seamlessly expand your images using the FLUX Klein 4B model combined with a specialized outpainting LoRA to outpaint images.
1. **Initialize:** Run Step 1 to prepare the environment.
2. **Download:** Run Step 2 to grab all necessary AI models.
3. **Generate:** Use Step 3 (Single) or Step 4 (Batch), set your padding sliders, run the cell, and upload your image(s). Your expanded images will download automatically!

In [ ]:
#@title 1. Initialize Core & Custom Nodes
#@markdown Run this cell first to set up the environment. It takes a few minutes, so grab a coffee!

import os
import subprocess

LOCAL_WORKSPACE = "/content/ComfyUI"

print("🚀 Initializing Core Architecture...")
if not os.path.exists(LOCAL_WORKSPACE):
    !git clone https://github.com/comfyanonymous/ComfyUI {LOCAL_WORKSPACE} &> /dev/null
    print("   ✓ Core Engine Cloned")
else:
    !cd {LOCAL_WORKSPACE} && git pull &> /dev/null
    print("   ✓ Core Engine Updated")

print("📦 Installing Dependencies (This takes a moment)...")
!cd {LOCAL_WORKSPACE} && pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 &> /dev/null

print("🧩 Installing Custom Nodes...")
RGTHREE_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/rgthree-comfy")
KJNODES_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-KJNodes")
ACE_PLUS_DIR = os.path.join(LOCAL_WORKSPACE, "custom_nodes/ComfyUI-ACE_Plus")

if not os.path.exists(RGTHREE_DIR):
    !git clone https://github.com/rgthree/rgthree-comfy {RGTHREE_DIR} &> /dev/null
    !pip install -r {RGTHREE_DIR}/requirements.txt &> /dev/null
    print("   ✓ RGThree Installed")

if not os.path.exists(KJNODES_DIR):
    !git clone https://github.com/kijai/ComfyUI-KJNodes {KJNODES_DIR} &> /dev/null
    !pip install -r {KJNODES_DIR}/requirements.txt &> /dev/null
    print("   ✓ KJNodes Installed")

if not os.path.exists(ACE_PLUS_DIR):
    !git clone https://github.com/ali-vilab/ComfyUI-ACE_Plus {ACE_PLUS_DIR} &> /dev/null
    !pip install -r {ACE_PLUS_DIR}/requirements.txt &> /dev/null
    print("   ✓ ACE_Plus Installed")

print("✅ Environment Ready!")

In [ ]:
#@title 2. High-Speed Asset Downloader
#@markdown Run this cell to download the necessary AI models and weights. Just click play and let it grab the files.

import os
import subprocess

WORKSPACE = "/content/ComfyUI"

MODELS = {
    "unet": [
        ("https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors", "flux-2-klein-4b.safetensors")
    ],
    "clip": [
        ("https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors", "qwen_3_4b.safetensors")
    ],
    "vae": [
        ("https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors", "flux2-vae.safetensors")
    ],
    "loras": [
        ("https://huggingface.co/fal/flux-2-klein-4B-outpaint-lora/resolve/main/LyNiaZ53Tudg0J6sT8Xbx_pytorch_lora_weights_comfy_converted.safetensors", "flux2_klein_4b_outpaint.safetensors")
    ]
}

DIRS = {
    "unet":           os.path.join(WORKSPACE, "models/diffusion_models"),
    "clip":           os.path.join(WORKSPACE, "models/text_encoders"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
}

print("⚡ Configuring Aria2c Accelerator...")
subprocess.run(['apt-get', '-y', 'install', '-qq', 'aria2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def download_file(url, target_dir, out_name):
    if not url.strip(): return
    print(f"   📥 Fetching: {out_name}...")
    try:
        aria2_cmd = [
            "aria2c", "--console-log-level=error", "--summary-interval=10",
            "-c", "-x", "16", "-s", "16", "-k", "1M",
            "--content-disposition=true", "--content-disposition-default-utf8=true",
            url, "-d", target_dir, "-o", out_name
        ]
        subprocess.run(aria2_cmd, check=True)
    except Exception as e:
        print(f"   ❌ Failed: {url}\n      Error: {e}\n")

def process_downloads(category):
    target_dir = DIRS[category]
    os.makedirs(target_dir, exist_ok=True)
    print(f"\n📂 Directory: {os.path.basename(target_dir)}")
    for url, out_name in MODELS[category]:
        download_file(url, target_dir, out_name)

process_downloads("unet")
process_downloads("loras")
process_downloads("clip")
process_downloads("vae")

print("\n✅ All assets secured with precise filenames!")

In [ ]:
#@title 3. Dynamic Single Outpainting Execution
#@markdown Adjust your padding settings below, run the cell, and upload your image when prompted. It will process and download your expanded image.

import subprocess
import time
import urllib.request
import urllib.parse
import json
import uuid
import os
import random
import shutil
from IPython.display import display, Image
from google.colab import files

# --- Colab Form Controls ---
MAX_BASE_DIMENSION = 1024 #@param {type:"slider", min:512, max:2048, step:64}
STEPS = 4 #@param {type:"slider", min:1, max:20, step:1}
SEED = 0 #@param {type:"integer"}
#@markdown ---
#@markdown **Padding Parameters (Pixels to add):**
PAD_LEFT = 300 #@param {type:"slider", min:0, max:1024, step:8}
PAD_RIGHT = 300 #@param {type:"slider", min:0, max:1024, step:8}
PAD_TOP = 0 #@param {type:"slider", min:0, max:1024, step:8}
PAD_BOTTOM = 0 #@param {type:"slider", min:0, max:1024, step:8}

WORKFLOW_URL = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/fluxkleinoutpaint.json"
PROMPT = "Fill the green spaces according to the image"

SERVER_ADDRESS = "127.0.0.1:8188"
WORKSPACE = "/content/ComfyUI"

try:
    import websocket
except ImportError:
    !pip install -q websocket-client
    import websocket

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

def queue_prompt(prompt, client_id):
    p = {"prompt": prompt, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    req =  urllib.request.Request(f"http://{SERVER_ADDRESS}/prompt", data=data)
    return json.loads(urllib.request.urlopen(req).read())

def get_image(filename, subfolder, folder_type):
    data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
    url_values = urllib.parse.urlencode(data)
    with urllib.request.urlopen(f"http://{SERVER_ADDRESS}/view?{url_values}") as response:
        return response.read()

def get_history(prompt_id):
    with urllib.request.urlopen(f"http://{SERVER_ADDRESS}/history/{prompt_id}") as response:
        return json.loads(response.read())

print("📤 Please upload the image you want to outpaint:")
uploaded = files.upload()

if not uploaded:
    print("❌ [ERROR] No image was uploaded.")
else:
    image_filename = list(uploaded.keys())[0]
    input_dir = os.path.join(WORKSPACE, "input")
    os.makedirs(input_dir, exist_ok=True)
    dest_path = os.path.join(input_dir, image_filename)
    shutil.move(image_filename, dest_path)
    print(f"📸 Image loaded: {image_filename}")

    server_ready = False
    try:
        urllib.request.urlopen(f"http://{SERVER_ADDRESS}/system_stats")
        server_ready = True
    except:
        print("🚀 Booting ComfyUI engine in the background...")
        log_file = open(f"{WORKSPACE}/comfy_log.txt", "w")
        subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd=WORKSPACE, stdout=log_file, stderr=subprocess.STDOUT)
        for i in range(120):
            try:
                urllib.request.urlopen(f"http://{SERVER_ADDRESS}/system_stats")
                server_ready = True
                break
            except:
                time.sleep(1)

    if not server_ready:
        print("\n❌ [ERROR] Engine failed to start.")
    else:
        print("🌐 Fetching workflow...")
        try:
            req = urllib.request.Request(WORKFLOW_URL)
            with urllib.request.urlopen(req) as response:
                workflow = json.loads(response.read())

            if "67" in workflow:
                workflow["67"]["inputs"]["unet_name"] = "flux-2-klein-4b.safetensors"
            if "69" in workflow:
                workflow["69"]["inputs"]["lora_1"]["lora"] = "flux2_klein_4b_outpaint.safetensors"
            if "6" in workflow:
                workflow["6"]["inputs"]["text"] = PROMPT
            if "7" in workflow:
                workflow["7"]["inputs"]["noise_seed"] = SEED
            if "45" in workflow:
                workflow["45"]["inputs"]["image"] = image_filename
            if "111" in workflow:
                workflow["111"]["inputs"]["largest_size"] = MAX_BASE_DIMENSION
            if "50" in workflow:
                workflow["50"]["inputs"]["left"] = PAD_LEFT
                workflow["50"]["inputs"]["right"] = PAD_RIGHT
                workflow["50"]["inputs"]["top"] = PAD_TOP
                workflow["50"]["inputs"]["bottom"] = PAD_BOTTOM
            if "8:17" in workflow:
                workflow["8:17"]["inputs"]["steps"] = STEPS

            client_id = str(uuid.uuid4())
            print("🚀 Processing generation...")
            prompt_id = queue_prompt(workflow, client_id)['prompt_id']

            ws = websocket.WebSocket()
            ws.connect(f"ws://{SERVER_ADDRESS}/ws?clientId={client_id}")
            while True:
                out = ws.recv()
                if isinstance(out, str):
                    message = json.loads(out)
                    if message['type'] == 'executing':
                        data = message['data']
                        if data['node'] is None and data['prompt_id'] == prompt_id:
                            break

            history = get_history(prompt_id)[prompt_id]
            for node_id in history['outputs']:
                if node_id == "9":
                    node_output = history['outputs'][node_id]
                    if 'images' in node_output:
                        for image in node_output['images']:
                            image_data = get_image(image['filename'], image['subfolder'], image['type'])
                            output_file = f"/content/outpainted_{image['filename']}"
                            with open(output_file, "wb") as f:
                                f.write(image_data)

                            print("🎉 Complete!")
                            display(Image(filename=output_file))
                            print("💾 Triggering file download...")
                            files.download(output_file)

        except Exception as e:
            print(f"\n❌ [ERROR]: {e}")

In [ ]:
#@title 4. Batch Outpainting Execution
#@markdown Adjust the uniform padding for your batch, run the cell, and upload multiple images. It will zip the expanded images and download them together.

import subprocess
import time
import urllib.request
import urllib.parse
import json
import uuid
import os
import random
import shutil
from google.colab import files

# --- Colab Form Controls ---
BATCH_MAX_BASE_DIMENSION = 1024 #@param {type:"slider", min:512, max:2048, step:64}
BATCH_STEPS = 4 #@param {type:"slider", min:1, max:20, step:1}
#@markdown ---
#@markdown **Uniform Padding Parameters:**
BATCH_PAD_LEFT = 300 #@param {type:"slider", min:0, max:1024, step:8}
BATCH_PAD_RIGHT = 300 #@param {type:"slider", min:0, max:1024, step:8}
BATCH_PAD_TOP = 0 #@param {type:"slider", min:0, max:1024, step:8}
BATCH_PAD_BOTTOM = 0 #@param {type:"slider", min:0, max:1024, step:8}

WORKFLOW_URL = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/fluxkleinoutpaint.json"
PROMPT = "Fill the green spaces according to the image"
SERVER_ADDRESS = "127.0.0.1:8188"
WORKSPACE = "/content/ComfyUI"
BATCH_OUT_DIR = "/content/batch_results_folder"

try:
    import websocket
except ImportError:
    !pip install -q websocket-client
    import websocket

def queue_prompt(prompt, client_id):
    p = {"prompt": prompt, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    req =  urllib.request.Request(f"http://{SERVER_ADDRESS}/prompt", data=data)
    return json.loads(urllib.request.urlopen(req).read())

def get_image(filename, subfolder, folder_type):
    data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
    url_values = urllib.parse.urlencode(data)
    with urllib.request.urlopen(f"http://{SERVER_ADDRESS}/view?{url_values}") as response:
        return response.read()

def get_history(prompt_id):
    with urllib.request.urlopen(f"http://{SERVER_ADDRESS}/history/{prompt_id}") as response:
        return json.loads(response.read())

print("📤 Please upload your batch of images:")
uploaded = files.upload()

if not uploaded:
    print("❌ [ERROR] No images uploaded for batch processing.")
else:
    os.makedirs(BATCH_OUT_DIR, exist_ok=True)
    input_dir = os.path.join(WORKSPACE, "input")
    os.makedirs(input_dir, exist_ok=True)

    server_ready = False
    try:
        urllib.request.urlopen(f"http://{SERVER_ADDRESS}/system_stats")
        server_ready = True
    except:
        print("🚀 Booting ComfyUI engine in the background...")
        log_file = open(f"{WORKSPACE}/comfy_log.txt", "w")
        subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd=WORKSPACE, stdout=log_file, stderr=subprocess.STDOUT)
        for i in range(120):
            try:
                urllib.request.urlopen(f"http://{SERVER_ADDRESS}/system_stats")
                server_ready = True
                break
            except:
                time.sleep(1)

    if not server_ready:
        print("\n❌ [ERROR] Engine failed to start.")
    else:
        try:
            req = urllib.request.Request(WORKFLOW_URL)
            with urllib.request.urlopen(req) as response:
                base_workflow = json.loads(response.read())

            for i, image_filename in enumerate(uploaded.keys()):
                print(f"\n🔄 Processing Image {i+1}/{len(uploaded)}: {image_filename}")
                dest_path = os.path.join(input_dir, image_filename)
                shutil.move(image_filename, dest_path)

                workflow = json.loads(json.dumps(base_workflow))

                if "67" in workflow:
                    workflow["67"]["inputs"]["unet_name"] = "flux-2-klein-4b.safetensors"
                if "69" in workflow:
                    workflow["69"]["inputs"]["lora_1"]["lora"] = "flux2_klein_4b_outpaint.safetensors"
                if "6" in workflow:
                    workflow["6"]["inputs"]["text"] = PROMPT
                if "7" in workflow:
                    workflow["7"]["inputs"]["noise_seed"] = random.randint(1, 1125899906842624)
                if "45" in workflow:
                    workflow["45"]["inputs"]["image"] = image_filename
                if "111" in workflow:
                    workflow["111"]["inputs"]["largest_size"] = BATCH_MAX_BASE_DIMENSION
                if "50" in workflow:
                    workflow["50"]["inputs"]["left"] = BATCH_PAD_LEFT
                    workflow["50"]["inputs"]["right"] = BATCH_PAD_RIGHT
                    workflow["50"]["inputs"]["top"] = BATCH_PAD_TOP
                    workflow["50"]["inputs"]["bottom"] = BATCH_PAD_BOTTOM
                if "8:17" in workflow:
                    workflow["8:17"]["inputs"]["steps"] = BATCH_STEPS

                client_id = str(uuid.uuid4())
                prompt_id = queue_prompt(workflow, client_id)['prompt_id']

                ws = websocket.WebSocket()
                ws.connect(f"ws://{SERVER_ADDRESS}/ws?clientId={client_id}")
                while True:
                    out = ws.recv()
                    if isinstance(out, str):
                        message = json.loads(out)
                        if message['type'] == 'executing':
                            data = message['data']
                            if data['node'] is None and data['prompt_id'] == prompt_id:
                                break

                history = get_history(prompt_id)[prompt_id]
                for node_id in history['outputs']:
                    if node_id == "9":
                        node_output = history['outputs'][node_id]
                        if 'images' in node_output:
                            for image in node_output['images']:
                                image_data = get_image(image['filename'], image['subfolder'], image['type'])
                                output_file = os.path.join(BATCH_OUT_DIR, f"outpainted_{image['filename']}")
                                with open(output_file, "wb") as f:
                                    f.write(image_data)
                                print(f"   ✓ Successfully saved to batch output.")

            print("\n📦 Zipping batch results...")
            zip_path = "/content/Batch_Outpaint_Results"
            shutil.make_archive(zip_path, 'zip', BATCH_OUT_DIR)

            print("💾 Triggering ZIP download...")
            files.download(f"{zip_path}.zip")

        except Exception as e:
            print(f"\n❌ [ERROR]: {e}")